# ML 실험 결과 종합 분석

## 노트북 목적
이 노트북은 03_generalization에서 수행한 **문항 조합 실험(6,096회)**과 **학생 분할 실험(360회)**의 결과를 종합적으로 분석합니다.

### 분석 관점
1. **성능 랭킹**: Kappa, F1 Macro, 합산 점수 기준으로 최고 성능 조합 식별
2. **문항 수 효과**: 학습 문항 수가 증가할수록 성능이 어떻게 변하는지
3. **교차 양식 효과**: 같은 양식(Form) vs 다른 양식에서의 일반화 차이
4. **문항별 기여도**: 각 문항이 모델 성능에 미치는 영향
5. **알고리즘 비교**: 24개 알고리즘의 상대적 성능
6. **학생 분할 비율**: 학습 데이터 비율에 따른 성능 변화

### 핵심 용어
- **Combined Score**: (Kappa + F1 Macro) / 2. 두 지표를 균형 있게 반영한 종합 점수
- **Same-Form**: 같은 시험 양식(a 또는 b) 내에서만 학습/테스트
- **Cross-Form**: 다른 시험 양식의 데이터로 테스트 (더 어려운 일반화 조건)
- **Leave-One-In**: 1개 문항만으로 학습하여 해당 문항의 독립 예측력 측정
- **Leave-One-Out**: 7개 문항으로 학습(1개 제외)하여 해당 문항의 기여도 측정

# ML 실험 결과 분석

문항 조합 실험(254개)과 학생 분할 실험(5비율×3seed)의 결과를 종합 분석합니다.

## 분석 관점
1. Kappa 기준 랭킹
2. F1 macro 기준 랭킹
3. 합산 (Kappa + F1) / 2 기준 랭킹
4. 문항 수별 성능 추이
5. Same-Form vs Cross-Form 효과
6. 문항별 독립 예측력 (Leave-One-In / Leave-One-Out)
7. 학생 분할 비율별 성능 변화

## 환경 설정
이 노트북은 분석 전용이므로 ML 라이브러리 없이 **pandas, numpy, matplotlib**만 사용합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 로드

03_generalization에서 생성된 CSV 결과 파일을 로드합니다.
- `all_results_question_combo.csv`: 254개 문항 조합 x 24개 알고리즘 = 6,096행
- `all_results_student_split.csv`: 5개 비율 x 3개 seed x 24개 알고리즘 = 360행

In [ ]:
qc_df = pd.read_csv('results/generalization/question_combo/all_results_question_combo.csv')
ss_df = pd.read_csv('results/generalization/student_split/all_results_student_split.csv')

print(f"문항 조합 결과: {qc_df.shape}")
print(f"학생 분할 결과: {ss_df.shape}")
print(f"\n문항 조합 컬럼: {qc_df.columns.tolist()}")
print(f"학생 분할 컬럼: {ss_df.columns.tolist()}")

## 2. 문항 조합 실험 분석

### 데이터 전처리
- **Combined Score** 계산: Kappa와 F1을 균등하게 반영한 종합 지표
- **양식 구성(Form Composition) 분류**: 학습 문항이 A양식만, B양식만, 또는 혼합인지 구분
  - `A_only`: 학습 문항이 모두 A양식 → 테스트는 B양식 (Cross-Form 일반화)
  - `B_only`: 학습 문항이 모두 B양식 → 테스트는 A양식
  - `Mixed`: 양식이 혼합 → 동일 양식 문항도 테스트에 포함 가능

In [ ]:
# Add combined metric
qc_df['Combined'] = (qc_df['Test_Kappa'] + qc_df['Test_F1']) / 2  # 두 지표의 균등 평균

# Add form composition analysis
def analyze_form_composition(train_qs_str):
    """Analyze how many A-form and B-form questions in the training set."""
    qs = train_qs_str.split(',')
    n_a = sum(1 for q in qs if q in ['Q1','Q2','Q3','Q4'])  # A양식 문항 수
    n_b = sum(1 for q in qs if q in ['Q5','Q6','Q7','Q8'])  # B양식 문항 수
    if n_b == 0:
        return 'A_only'
    elif n_a == 0:
        return 'B_only'
    else:
        return 'Mixed'

qc_df['Form_Comp'] = qc_df['Train_Qs'].apply(analyze_form_composition)
qc_df['N_A'] = qc_df['Train_Qs'].apply(lambda x: sum(1 for q in x.split(',') if q in ['Q1','Q2','Q3','Q4']))
qc_df['N_B'] = qc_df['Train_Qs'].apply(lambda x: sum(1 for q in x.split(',') if q in ['Q5','Q6','Q7','Q8']))

### 2.1 3가지 기준 랭킹

Kappa, F1 Macro, Combined Score 3가지 기준으로 상위 20개 조합을 확인합니다.
각 기준에서 일관되게 높은 순위를 보이는 조합이 가장 신뢰할 수 있는 조합입니다.

In [ ]:
# Top 20 by each criterion
for criterion, col in [('Kappa', 'Test_Kappa'), ('F1 Macro', 'Test_F1'), ('Combined', 'Combined')]:
    print(f"\n{'='*90}")
    print(f"상위 20개 (기준: {criterion})")
    print(f"{'='*90}")
    top = qc_df.nlargest(20, col)
    print(top[['Combo', 'N_Train_Qs', 'Form_Comp', 'Model', 'Type', 
               'Test_Kappa', 'Test_F1', 'Combined', 'CV_Kappa']].to_string(index=False))

# Save rankings
for name, col in [('kappa', 'Test_Kappa'), ('f1macro', 'Test_F1'), ('combined', 'Combined')]:
    ranked = qc_df.sort_values(col, ascending=False)
    ranked.to_csv(f'results/generalization/question_combo/ranking_by_{name}.csv', index=False)
    print(f"\n저장: ranking_by_{name}.csv")

### 2.2 문항 수별 성능 추이

### 문항 수와 성능의 관계
학습에 사용하는 문항 수(1~7개)에 따라 성능이 어떻게 변화하는지 박스플롯으로 시각화합니다.
- **박스**: 사분위 범위 (중간 50% 데이터)
- **빨간 선**: 평균 성능
- **초록 점선**: 최고 성능

일반적으로 문항 수가 증가할수록 학습 데이터가 많아져 성능이 향상되지만, 7문항(학습)→1문항(테스트)은 테스트 데이터가 너무 적어 불안정할 수 있습니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (metric, label) in enumerate([('Test_Kappa', 'Kappa'), ('Test_F1', 'F1 Macro'), ('Combined', 'Combined')]):
    ax = axes[idx]
    
    # Box plot by N_Train_Qs
    data_by_k = [qc_df[qc_df['N_Train_Qs']==k][metric].values for k in range(1,8)]  # 문항 수별 성능 분포
    bp = ax.boxplot(data_by_k, labels=range(1,8), patch_artist=True)
    
    # Mean line
    means = [qc_df[qc_df['N_Train_Qs']==k][metric].mean() for k in range(1,8)]
    ax.plot(range(1,8), means, 'r-o', linewidth=2, label='Mean')
    
    # Best per k
    bests = [qc_df[qc_df['N_Train_Qs']==k][metric].max() for k in range(1,8)]
    ax.plot(range(1,8), bests, 'g--^', linewidth=1.5, label='Best')
    
    ax.set_xlabel('Train 문항 수')
    ax.set_ylabel(label)
    ax.set_title(f'{label} by Train Questions')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/generalization/question_combo/fig_performance_by_nq.png', dpi=150, bbox_inches='tight')
plt.show()

# Print statistics
print("\n문항 수별 Test Kappa 통계:")
stats = qc_df.groupby('N_Train_Qs')['Test_Kappa'].agg(['mean','std','min','max','count'])
stats.columns = ['Mean', 'Std', 'Min', 'Max', 'N_experiments']
print(stats.round(4))

**결과 해석:** 문항 수별 성능 통계에서 평균(Mean)과 표준편차(Std)를 확인합니다.
문항 수가 증가할수록 평균 Kappa가 향상되는 경향이 있으며, 이는 더 많은 학습 데이터가 모델 성능을 개선함을 보여줍니다.

### 2.3 Same-Form vs Cross-Form 효과

**Same-Form vs Cross-Form 분석의 의미:**
학습 데이터와 테스트 데이터가 같은 양식(Form)에서 오는지, 다른 양식에서 오는지에 따라 성능이 어떻게 달라지는지 확인합니다.

- `Mixed` (혼합): 가장 좋은 성능이 예상됨 (다양한 양식의 피드백을 학습)
- `A_only` 또는 `B_only`: Cross-Form 일반화 능력을 테스트 (더 도전적)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# By form composition
for idx, metric in enumerate(['Test_Kappa', 'Test_F1']):
    ax = axes[idx]
    for comp in ['A_only', 'B_only', 'Mixed']:
        subset = qc_df[qc_df['Form_Comp'] == comp]
        means = subset.groupby('N_Train_Qs')[metric].mean()
        ax.plot(means.index, means.values, 'o-', label=comp, linewidth=2)
    
    ax.set_xlabel('Train 문항 수')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Form Composition')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/generalization/question_combo/fig_form_composition.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistical comparison
print("\n=== Form 구성별 평균 성능 ===")
comp_stats = qc_df.groupby(['N_Train_Qs', 'Form_Comp']).agg({
    'Test_Kappa': 'mean', 'Test_F1': 'mean'
}).round(4)
print(comp_stats)

**결과 해석:** Mixed 구성이 A_only/B_only보다 높은 성능을 보이면, 양식 간 일반화가 어렵다는 것을 의미합니다.
반대로 차이가 작다면, 피드백의 특성이 양식에 관계없이 일관적임을 시사합니다.

### 2.4 문항별 독립 예측력 (Leave-One-In)

**Leave-One-In 분석:** 각 문항 하나만으로 학습했을 때의 성능을 비교합니다.
성능이 높은 문항은 피드백 패턴의 다양성이 풍부하여 독립적 예측력이 높다는 의미입니다.

In [ ]:
# 1-question combos: each question's independent predictive power
one_q = qc_df[qc_df['N_Train_Qs'] == 1].copy()

print("=== 문항별 독립 예측력 (1문항 Train) ===")
print(f"{'문항':>6} {'Mean Kappa':>12} {'Best Kappa':>12} {'Mean F1':>10} {'Best Model':>30}")
print("-" * 80)

for q in ['Q1','Q2','Q3','Q4','Q5','Q6','Q7','Q8']:
    subset = one_q[one_q['Combo'] == q]
    if len(subset) > 0:
        best = subset.loc[subset['Test_Kappa'].idxmax()]
        print(f"{q:>6} {subset['Test_Kappa'].mean():>12.4f} {best['Test_Kappa']:>12.4f} {subset['Test_F1'].mean():>10.4f} {best['Model']:>30}")

### 2.5 문항별 기여도 (Leave-One-Out)

**Leave-One-Out 분석:** 7개 문항으로 학습하고 제외된 1개 문항으로 테스트합니다.
성능 저하가 큰 문항은 다른 문항들과 다른 특성을 가져 일반화가 어려운 문항입니다.

In [ ]:
# 7-question combos: impact of removing each question
seven_q = qc_df[qc_df['N_Train_Qs'] == 7].copy()

print("=== 문항별 기여도 (7문항 Train, 1문항 제외) ===")
print(f"{'제외 문항':>10} {'Mean Kappa':>12} {'Best Kappa':>12} {'Mean F1':>10} {'Best Model':>30}")
print("-" * 85)

for q in ['Q1','Q2','Q3','Q4','Q5','Q6','Q7','Q8']:
    # Find combos that DON'T include this question
    subset = seven_q[~seven_q['Train_Qs'].str.contains(q)]
    if len(subset) > 0:
        best = subset.loc[subset['Test_Kappa'].idxmax()]
        print(f"{q:>10} {subset['Test_Kappa'].mean():>12.4f} {best['Test_Kappa']:>12.4f} {subset['Test_F1'].mean():>10.4f} {best['Model']:>30}")

### 2.6 알고리즘별 성능 비교

24개 알고리즘의 성능을 문항 수별로 비교하여, 어떤 알고리즘이 데이터 양에 관계없이 안정적인지 확인합니다.
히트맵에서 전체적으로 진한 색을 보이는 알고리즘이 가장 안정적이고 우수한 알고리즘입니다.

In [ ]:
# Best algorithm per N_Train_Qs
print("=== 문항 수별 최고 알고리즘 ===\n")
for k in range(1, 8):
    subset = qc_df[qc_df['N_Train_Qs'] == k]
    
    # Average across all combos for each algorithm
    algo_mean = subset.groupby('Model').agg({
        'Test_Kappa': 'mean', 'Test_F1': 'mean'
    }).round(4)
    
    best_kappa = algo_mean['Test_Kappa'].idxmax()
    best_f1 = algo_mean['Test_F1'].idxmax()
    
    print(f"[{k}문항] Kappa 최고: {best_kappa} ({algo_mean.loc[best_kappa, 'Test_Kappa']:.4f}), "
          f"F1 최고: {best_f1} ({algo_mean.loc[best_f1, 'Test_F1']:.4f})")

# Heatmap: Algorithm x N_Train_Qs
pivot = qc_df.groupby(['N_Train_Qs', 'Model'])['Test_Kappa'].mean().unstack()

fig, ax = plt.subplots(figsize=(20, 8))
im = ax.imshow(pivot.T.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.index)
ax.set_yticks(range(len(pivot.columns)))
ax.set_yticklabels(pivot.columns, fontsize=8)
ax.set_xlabel('Train 문항 수')
ax.set_ylabel('Algorithm')
ax.set_title('Mean Test Kappa: Algorithm \u00d7 Train Questions')
plt.colorbar(im)
plt.tight_layout()
plt.savefig('results/generalization/question_combo/fig_algo_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

**결과 해석:** 알고리즘별 히트맵에서 행(알고리즘)과 열(문항 수)의 교차점 색상으로 성능을 비교합니다.
특정 문항 수에서만 좋은 성능을 보이는 알고리즘보다, 모든 조건에서 고르게 높은 성능을 보이는 알고리즘이 더 실용적입니다.

## 3. 학생 분할 실험 분석

### 학생 분할 실험 분석
학생 단위로 분할한 실험 결과를 분석합니다.
학습 데이터 비율(80:20 ~ 40:60)에 따른 성능 변화와, 개별 모델 vs 앙상블의 성능 차이를 확인합니다.

**Seed 안정성**: 동일 비율에서 3개 seed의 성능 표준편차가 작을수록 결과가 안정적입니다.

In [ ]:
ss_df['Combined'] = (ss_df['Test_Kappa'] + ss_df['Test_F1']) / 2  # 학생 분할에도 동일한 합산 지표 적용

# Rankings
for criterion, col in [('Kappa', 'Test_Kappa'), ('F1 Macro', 'Test_F1'), ('Combined', 'Combined')]:
    print(f"\n{'='*90}")
    print(f"학생 분할 상위 10개 (기준: {criterion})")
    print(f"{'='*90}")
    top = ss_df.nlargest(10, col)
    print(top[['Ratio', 'Seed', 'Model', 'Type',
               'Test_Kappa', 'Test_F1', 'Combined', 'Train_N', 'Test_N']].to_string(index=False))

# Save rankings
for name, col in [('kappa', 'Test_Kappa'), ('f1macro', 'Test_F1'), ('combined', 'Combined')]:
    ranked = ss_df.sort_values(col, ascending=False)
    ranked.to_csv(f'results/generalization/student_split/ranking_by_{name}.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (metric, label) in enumerate([('Test_Kappa', 'Kappa'), ('Test_F1', 'F1 Macro'), ('Combined', 'Combined')]):
    ax = axes[idx]
    
    # Average across seeds for each ratio
    ratio_order = ['80:20', '70:30', '60:40', '50:50', '40:60']
    
    # Individual vs Ensemble
    for mtype, style in [('Individual', 'o-'), ('Ensemble', 's--')]:
        subset = ss_df[ss_df['Type'] == mtype]
        means = subset.groupby('Ratio')[metric].mean()
        means = means.reindex(ratio_order)
        ax.plot(range(len(ratio_order)), means.values, style, label=mtype, linewidth=2)
    
    ax.set_xticks(range(len(ratio_order)))
    ax.set_xticklabels(ratio_order)
    ax.set_xlabel('Train:Test 비율')
    ax.set_ylabel(label)
    ax.set_title(f'{label} by Student Split Ratio')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/generalization/student_split/fig_ratio_performance.png', dpi=150, bbox_inches='tight')
plt.show()

# Seed stability
print("\n=== Seed간 안정성 (각 비율에서 3 seed의 표준편차) ===")
seed_stability = ss_df.groupby(['Ratio', 'Model']).agg({
    'Test_Kappa': ['mean', 'std'],
    'Test_F1': ['mean', 'std']
})
seed_stability.columns = ['Kappa_Mean', 'Kappa_Std', 'F1_Mean', 'F1_Std']
print(seed_stability.groupby('Ratio').mean().round(4))

**결과 해석:** 
- 학습 비율이 감소해도 성능이 크게 떨어지지 않으면, 모델이 적은 데이터로도 일반화가 가능함을 의미합니다.
- 앙상블(Ensemble)이 개별(Individual) 모델보다 일관적으로 높은 성능을 보이는지 확인합니다.
- Seed 간 표준편차가 작을수록 랜덤 분할에 대해 견고한 모델입니다.

## 4. 종합 요약

### 종합 분석 결과
문항 조합 실험과 학생 분할 실험의 핵심 발견을 정리합니다.

주요 확인 사항:
1. 최고 Kappa를 달성한 문항 조합과 알고리즘
2. 문항 수에 따른 성능 향상 추이
3. 학습 비율에 따른 성능 변화
4. 전체 최적 ML 모델과 조건 조합

In [ ]:
print("=" * 90)
print("종합 분석 요약")
print("=" * 90)

# Question combo summary
print("\n[문항 조합 실험]")
print(f"  총 실험: {len(qc_df)}회 (254 조합 \u00d7 24 알고리즘)")
print(f"  전체 최고 Kappa: {qc_df['Test_Kappa'].max():.4f}")
best_overall = qc_df.loc[qc_df['Test_Kappa'].idxmax()]
print(f"    \u2192 조합: {best_overall['Combo']}, 모델: {best_overall['Model']}")

print(f"\n  문항 수별 평균 Kappa:")
for k in range(1, 8):
    mean_k = qc_df[qc_df['N_Train_Qs']==k]['Test_Kappa'].mean()
    max_k = qc_df[qc_df['N_Train_Qs']==k]['Test_Kappa'].max()
    print(f"    {k}문항: 평균 {mean_k:.4f}, 최고 {max_k:.4f}")

# Student split summary
print(f"\n[학생 분할 실험]")
print(f"  총 실험: {len(ss_df)}회 (5 비율 \u00d7 3 seed \u00d7 24 알고리즘)")
print(f"  전체 최고 Kappa: {ss_df['Test_Kappa'].max():.4f}")
best_ss = ss_df.loc[ss_df['Test_Kappa'].idxmax()]
print(f"    \u2192 비율: {best_ss['Ratio']}, Seed: {best_ss['Seed']}, 모델: {best_ss['Model']}")

print(f"\n  비율별 평균 Kappa:")
for ratio in ['80:20', '70:30', '60:40', '50:50', '40:60']:
    subset = ss_df[ss_df['Ratio']==ratio]
    if len(subset) > 0:
        print(f"    {ratio}: 평균 {subset['Test_Kappa'].mean():.4f}, 최고 {subset['Test_Kappa'].max():.4f}")

print(f"\n완료: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")